# 02 Data Cleaning

**Input:** `data/interim/01_user_stories_raw.csv`. This file has 31,394 stories pulled raw from MySQL in notebook 01.

**Goal:** turn the raw text into clean strings without dropping rows. We keep missing values and empty fields, because in this project missingness is itself a quality signal (see decision D5 in `docs/decisions.md`).

**References used in this notebook:**
- Wickham (2016), Tidy Data principles. One row per story, one attribute per column, missing values stay explicit.
- Thong et al. (2017), data capture-ability versus data analyzability. We move records from "captured but messy" to "captured and analyzable", without deleting any of them.
- Lucassen et al. (2016), QUS framework. Cleaning supports the syntactic criteria "Full sentence" and "Minimal".

**Output:** `data/processed/02_user_stories_clean.csv`.

## Plan

1. Load the raw CSV.
2. Profile the artifacts. Count exactly how many stories carry quotes, triple quotes, HTML, control characters, and so on.
3. Write small cleaning functions, one per real problem.
4. Apply them to Title and Description text.
5. Check for exact duplicates.
6. Flag outliers in Story Point and text length. Flag only, do not drop.
7. Run a few sanity checks at the end.
8. Save the cleaned CSV.

## Notes

- Numeric columns such as Story_Point are not modified here. They are handled in notebook 03 (feature engineering).
- Duplicate handling is conservative. Only exact "Title plus Description_Text" duplicates within the same project are flagged.
- Near duplicate detection (similar but not identical) is left for a later notebook.

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)
RAW_PATH = Path('../data/interim/01_user_stories_raw.csv')
df = pd.read_csv(RAW_PATH, low_memory=False)

for col in ['Creation_Date', 'Resolution_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"Loaded: {RAW_PATH}")
print(f"Shape:  {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

df.head(3)

Loaded: ..\data\interim\01_user_stories_raw.csv
Shape:  (31394, 14)
Memory: 27.8 MB


,ID,Issue_Key,Project_Name,Title,Description,Description_Text,Type,Status,Resolution,Story_Point,Creation_Date,Resolution_Date,Resolution_Time_Minutes,Story_Point_Changed_After_Estimation
0,68,XD-3765,Spring XD,"""Fix stream failover ""","""See https://github.com/spring-projects/spring-xd/issues/1924""","""""""See https://github.com/spring-projects/spring-xd/issues/1924""""""",Story,Done,Complete,8.0,2017-03-21 16:54:44,2017-03-21 16:55:23,0.0,0
1,77,XD-3756,Spring XD,"""wordcount failed to run in cloudera VM 5.7""","""I download the spring XD example projects, and run through the steps acccording the README file for the project. I tried to change the hadoop-site.xml, server.yml and wordcount.xml files, but I f...","""""""I download the spring XD example projects, and run through the steps acccording the README file for the project. I tried to change the hadoop-site.xml, server.yml and wordcount.xml files, but I...",Story,To Do,NaN,3.0,2016-05-23 10:11:59,NaT,0.0,0
2,109,XD-3724,Spring XD,"""Add Job RDBMS config in Ambari plugin""","""Spring XD Ambari plugin only supports HDB as job db. HDB is not good in production environment. It will be great if we can specify RDB in spring xd installation/config process. ""","""""""Spring XD Ambari plugin only supports HDB as job db. HDB is not good in production environment. It will be great if we can specify RDB in spring xd installation/config process. """"""",Story,To Do,NaN,2.0,2015-12-16 18:38:51,NaT,0.0,0


In [2]:
# First step,Artifact profiling
# We count how many stories carry each type of dirty text.
# We do not fix anything here. We only measure. This tells us which cleaning rules are worth writing.

title = df['Title'].fillna('')
desc  = df['Description_Text'].fillna('')
profile = {}

profile['title_wrapped_in_quotes'] = (
    title.str.startswith('"') & title.str.endswith('"')
).sum()

profile['desc_starts_with_triple_quotes'] = (
    desc.str.startswith('"""')
).sum()

profile['desc_wrapped_in_single_quotes'] = (
    desc.str.startswith('"')
    & desc.str.endswith('"')
    & ~desc.str.startswith('"""')
).sum()

profile['title_has_edge_whitespace'] = (
    title.str.strip() != title
).sum()
profile['desc_has_edge_whitespace'] = (
    desc.str.strip() != desc
).sum()

html_pattern = re.compile(r'<[a-zA-Z/][^>]*>')
profile['title_has_html'] = title.str.contains(html_pattern, na=False).sum()
profile['desc_has_html']  = desc.str.contains(html_pattern, na=False).sum()
profile['desc_has_jira_markup'] = desc.str.contains(
    r'\{code[^\}]*\}|\{noformat\}|\{quote\}|\{panel[^\}]*\}',
    regex=True, na=False
).sum()

profile['desc_has_url'] = desc.str.contains(r'https?://', regex=True, na=False).sum()
control_pattern = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')
profile['title_has_control_chars'] = title.str.contains(control_pattern, na=False).sum()
profile['desc_has_control_chars']  = desc.str.contains(control_pattern, na=False).sum()
profile['title_empty_after_strip'] = (title.str.strip() == '').sum()
profile['desc_empty_after_strip']  = (desc.str.strip() == '').sum()
total = len(df)
print(f"Total stories: {total:,}\n")
print(f"{'Artifact':<38} {'Count':>10} {'Share':>8}")
print("-" * 60)
for key, count in profile.items():
    pct = count / total * 100
    print(f"{key:<38} {count:>10,} {pct:>7.1f}%")

Total stories: 31,394

Artifact                                    Count    Share
------------------------------------------------------------
title_wrapped_in_quotes                    31,394   100.0%
desc_starts_with_triple_quotes             27,327    87.0%
desc_wrapped_in_single_quotes                   0     0.0%
title_has_edge_whitespace                       0     0.0%
desc_has_edge_whitespace                        0     0.0%
title_has_html                                 40     0.1%
desc_has_html                               2,389     7.6%
desc_has_jira_markup                          415     1.3%
desc_has_url                                4,211    13.4%
title_has_control_chars                         0     0.0%
desc_has_control_chars                          5     0.0%
title_empty_after_strip                         0     0.0%
desc_empty_after_strip                      4,067    13.0%


In [3]:
# Cleaning function for Title
# What this fixes (based on the profile we just ran):
#   1. Title is wrapped in double quotes for every single row.
#   2. There may be extra spaces around the actual text.
#   3. Rare HTML tags (40 rows) should be stripped to plain text.
# What this does NOT do:
#   - It does not lowercase, because case can carry meaning in technical titles.
#   - It does not drop empty results. An empty title after cleaning stays empty.
HTML_TAG_RE = re.compile(r'<[a-zA-Z/][^>]*>')

def clean_title(text):
    """
    Take one raw title string. Return a cleaned title.
    Steps are applied in this order:
      1. Strip outer whitespace.
      2. If wrapped in matching double quotes, remove them.
      3. Strip whitespace again (the inner edge may now be exposed).
      4. Remove HTML tags if any.
      5. Collapse runs of whitespace into a single space.
    """
    if pd.isna(text):
        return text
    s = str(text).strip()
    if len(s) >= 2 and s.startswith('"') and s.endswith('"'):
        s = s[1:-1].strip()

    s = HTML_TAG_RE.sub(' ', s)
    s = re.sub(r'\s+', ' ', s).strip()

    return s

sample = df['Title'].head(5).tolist()
print("Before  -> After")
print("-" * 80)
for raw in sample:
    cleaned = clean_title(raw)
    print(f"{raw!r}")
    print(f"  -> {cleaned!r}")
    print()

Before  -> After
--------------------------------------------------------------------------------
'"Fix stream failover "'
  -> 'Fix stream failover'

'"wordcount failed to run in cloudera VM 5.7"'
  -> 'wordcount failed to run in cloudera VM 5.7'

'"Add Job RDBMS config in Ambari plugin"'
  -> 'Add Job RDBMS config in Ambari plugin'

'"Move k8s SPI to a separate repo"'
  -> 'Move k8s SPI to a separate repo'

'"Upgrade XD Ambari release to 1.3 "'
  -> 'Upgrade XD Ambari release to 1.3'



In [4]:
# Apply clean_title to the whole column.
# I create a new column Title_Clean. The original Title stays untouched.
df['Title_Clean'] = df['Title'].apply(clean_title)
print(f"Rows processed: {len(df):,}")
print(f"Title_Clean non null:    {df['Title_Clean'].notna().sum():,}")
print(f"Title_Clean empty string: {(df['Title_Clean'] == '').sum():,}\n")
df['_title_len_raw']   = df['Title'].fillna('').str.len()
df['_title_len_clean'] = df['Title_Clean'].fillna('').str.len()
mean_chars_removed = (df['_title_len_raw'] - df['_title_len_clean']).mean()
print(f"Average characters removed per title: {mean_chars_removed:.2f}")
print("(Most of this is the two wrapping quotes plus stray whitespace.)\n")
print("Sample before vs after:")
print("-" * 80)
print(df[['Title', 'Title_Clean']].head(5).to_string(index=False))
df = df.drop(columns=['_title_len_raw', '_title_len_clean'])

Rows processed: 31,394
Title_Clean non null:    31,394
Title_Clean empty string: 0

Average characters removed per title: 2.06
(Most of this is the two wrapping quotes plus stray whitespace.)

Sample before vs after:
--------------------------------------------------------------------------------
                                       Title                                Title_Clean
                      "Fix stream failover "                        Fix stream failover
"wordcount failed to run in cloudera VM 5.7" wordcount failed to run in cloudera VM 5.7
     "Add Job RDBMS config in Ambari plugin"      Add Job RDBMS config in Ambari plugin
           "Move k8s SPI to a separate repo"            Move k8s SPI to a separate repo
         "Upgrade XD Ambari release to 1.3 "           Upgrade XD Ambari release to 1.3


In [5]:
# Cleaning function for Description
#   1. Triple double quotes wrapping the text. Example: """abc""" becomes abc
#   2. Single double quotes wrapping the text. Example: "abc" becomes abc
#   3. HTML tags. Example: <p>hello</p> becomes hello
#   4. Jira specific markup. Example: {code}print(x){code} is removed.
#   5. Control characters such as null bytes or stray non printable bytes.
#   6. Multiple whitespaces collapsed into a single space.
#   - It does not remove URLs. URLs are content, not artifacts.
#   - It does not change case.
#   - It does not drop empty results. An empty description after cleaning
#     stays empty. Per decision D5, empty description is itself a quality
#     signal.
#   - It does not touch line breaks aggressively. We collapse runs of
#     whitespace, but a paragraph break loses no real meaning here.
TRIPLE_QUOTE_OUTER = re.compile(r'^"{3}(.*?)"{3}$', re.DOTALL)
SINGLE_QUOTE_OUTER = re.compile(r'^"(.*?)"$',     re.DOTALL)
HTML_TAG_RE        = re.compile(r'<[a-zA-Z/][^>]*>')
JIRA_MARKUP_RE     = re.compile(
    r'\{code[^\}]*\}.*?\{code\}'
    r'|\{noformat\}.*?\{noformat\}'
    r'|\{quote\}.*?\{quote\}'
    r'|\{panel[^\}]*\}.*?\{panel\}',
    re.DOTALL
)
CONTROL_CHARS_RE   = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f]')
WHITESPACE_RUN     = re.compile(r'\s+')
def clean_description(text):
    """
    Take one raw description string. Return a cleaned description.
    """
    if pd.isna(text):
        return text
    s = str(text).strip()
    m = TRIPLE_QUOTE_OUTER.match(s)
    if m is not None:
        s = m.group(1).strip()
    else:
        m = SINGLE_QUOTE_OUTER.match(s)
        if m is not None:
            s = m.group(1).strip()

    s = JIRA_MARKUP_RE.sub(' ', s)
    s = HTML_TAG_RE.sub(' ', s)
    s = CONTROL_CHARS_RE.sub('', s)
    s = WHITESPACE_RUN.sub(' ', s).strip()

    return s
sample_idx = [0, 1, 2, 3, 4]
print("Before vs after on a small sample:")
print("=" * 80)
for i in sample_idx:
    raw = df.loc[i, 'Description_Text']
    cleaned = clean_description(raw)
    raw_str = str(raw)
    cleaned_str = str(cleaned)
    print(f"Row {i}:")
    print(f"  Raw     ({len(raw_str):>4} chars): {raw_str[:150]}{'...' if len(raw_str) > 150 else ''}")
    print(f"  Cleaned ({len(cleaned_str):>4} chars): {cleaned_str[:150]}{'...' if len(cleaned_str) > 150 else ''}")
    print()

Before vs after on a small sample:
Row 0:
  Raw     (  66 chars): """See https://github.com/spring-projects/spring-xd/issues/1924"""
  Cleaned (  60 chars): See https://github.com/spring-projects/spring-xd/issues/1924

Row 1:
  Raw     ( 299 chars): """I download the spring XD example projects, and run through the steps acccording the README file for the project. I tried to change the hadoop-site....
  Cleaned ( 293 chars): I download the spring XD example projects, and run through the steps acccording the README file for the project. I tried to change the hadoop-site.xml...

Row 2:
  Raw     ( 184 chars): """Spring XD Ambari plugin only supports HDB as job db. HDB is not good in production environment. It will be great if we can specify RDB in spring xd...
  Cleaned ( 176 chars): Spring XD Ambari plugin only supports HDB as job db. HDB is not good in production environment. It will be great if we can specify RDB in spring xd in...

Row 3:
  Raw     (  64 chars): """As a developer, I'd

In [6]:
df['Description_Clean'] = df['Description_Text'].apply(clean_description)
n_total       = len(df)
n_non_null    = df['Description_Clean'].notna().sum()
n_empty_str   = (df['Description_Clean'].fillna('') == '').sum()
n_was_empty   = (df['Description_Text'].fillna('').str.strip() == '').sum()
print(f"Rows processed:                        {n_total:,}")
print(f"Description_Clean non null:            {n_non_null:,}")
print(f"Description_Clean empty after clean:   {n_empty_str:,}")
print(f"Description_Text was empty before:     {n_was_empty:,}")
print()
new_empties = n_empty_str - n_was_empty
print(f"New empty descriptions after cleaning: {new_empties:,}")
print("(These are stories that contained only markup, no actual prose.)\n")
raw_lens   = df['Description_Text'].fillna('').str.len()
clean_lens = df['Description_Clean'].fillna('').str.len()
removed    = (raw_lens - clean_lens)
print("Characters removed per description:")
print(f"  mean:   {removed.mean():.1f}")
print(f"  median: {removed.median():.1f}")
print(f"  max:    {removed.max():,}")
print()
print("(Mean is dominated by the 6 char triple quote wrap. Higher numbers")
print(" point to descriptions with embedded HTML or Jira code blocks.)")

Rows processed:                        31,394
Description_Clean non null:            27,327
Description_Clean empty after clean:   4,102
Description_Text was empty before:     4,067

New empty descriptions after cleaning: 35
(These are stories that contained only markup, no actual prose.)

Characters removed per description:
  mean:   20.2
  median: 7.0
  max:    9,154

(Mean is dominated by the 6 char triple quote wrap. Higher numbers
 point to descriptions with embedded HTML or Jira code blocks.)


In [7]:
# Duplicate check
dup_key = (
    df['Project_Name'].fillna('')
    + '||'
    + df['Title_Clean'].fillna('')
    + '||'
    + df['Description_Clean'].fillna('')
)

df['is_duplicate_in_project'] = dup_key.duplicated(keep=False)
n_dups = df['is_duplicate_in_project'].sum()
n_unique_dup_keys = dup_key[df['is_duplicate_in_project']].nunique()

print(f"Rows that share their full text with another row in the same project: {n_dups:,}")
print(f"Number of distinct duplicate groups:                                  {n_unique_dup_keys:,}")
print(f"Average duplicates per group:                                         {n_dups / n_unique_dup_keys:.2f}" if n_unique_dup_keys else "")
print()
if n_dups > 0:
    example_key = dup_key[df['is_duplicate_in_project']].iloc[0]
    example_group = df[dup_key == example_key][
        ['Issue_Key', 'Project_Name', 'Title_Clean', 'Status', 'Story_Point', 'Creation_Date']
    ]
    print("One example duplicate group:")
    print(example_group.to_string(index=False))
else:
    print("No exact duplicates found.")

Rows that share their full text with another row in the same project: 540
Number of distinct duplicate groups:                                  164
Average duplicates per group:                                         3.29

One example duplicate group:
Issue_Key Project_Name                                                  Title_Clean Status  Story_Point       Creation_Date
  XD-2186    Spring XD Fix 'cluster/containers' REST endpoint with security enabled   Done          3.0 2014-09-26 10:47:52
  XD-2185    Spring XD Fix 'cluster/containers' REST endpoint with security enabled   Done          3.0 2014-09-26 10:47:33
  XD-2184    Spring XD Fix 'cluster/containers' REST endpoint with security enabled   Done          3.0 2014-09-26 10:45:04
  XD-2183    Spring XD Fix 'cluster/containers' REST endpoint with security enabled  To Do          3.0 2014-09-26 10:44:33


In [8]:
# Outlier and quality flags
df['title_char_count']       = df['Title_Clean'].fillna('').str.len()
df['title_word_count']       = df['Title_Clean'].fillna('').str.split().str.len()
df['description_char_count'] = df['Description_Clean'].fillna('').str.len()
df['description_word_count'] = df['Description_Clean'].fillna('').str.split().str.len()
df['flag_title_too_short']        = df['title_word_count'] < 3
df['flag_title_too_long']         = df['title_word_count'] > 20
df['flag_description_missing']    = df['description_char_count'] == 0
df['flag_description_too_short']  = (
    (df['description_word_count'] > 0) & (df['description_word_count'] < 10)
)

df['flag_description_markup_only'] = (
    (df['Description_Text'].fillna('').str.strip() != '')
    & (df['description_char_count'] == 0)
)
fibonacci = {0, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144}

df['flag_sp_non_fibonacci'] = (
    df['Story_Point'].notna()
    & ~df['Story_Point'].isin(fibonacci)
)
df['flag_sp_high_scope_risk'] = df['Story_Point'] >= 13
df['flag_sp_extreme_scope_risk'] = df['Story_Point'] >= 40
df['flag_sp_missing'] = df['Story_Point'].isna()
df['flag_duplicate_in_project'] = df['is_duplicate_in_project']

flag_cols = [c for c in df.columns if c.startswith('flag_')]

print(f"{'Flag':<38} {'True count':>12} {'Share':>10}")
print("-" * 65)
for col in flag_cols:
    n = int(df[col].sum())
    pct = n / len(df) * 100
    print(f"{col:<38} {n:>12,} {pct:>9.1f}%")

Flag                                     True count      Share
-----------------------------------------------------------------
flag_title_too_short                            845       2.7%
flag_title_too_long                             207       0.7%
flag_description_missing                      4,102      13.1%
flag_description_too_short                    2,281       7.3%
flag_description_markup_only                     35       0.1%
flag_sp_non_fibonacci                         7,500      23.9%
flag_sp_high_scope_risk                       1,094       3.5%
flag_sp_extreme_scope_risk                       75       0.2%
flag_sp_missing                               9,661      30.8%
flag_duplicate_in_project                       540       1.7%


In [22]:
# Sanity checks
assert len(df) == 31394, f"Row count changed: {len(df)}"
assert 'Title_Clean'       in df.columns
assert 'Description_Clean' in df.columns
assert df['Title_Clean'].notna().all(), "Some Title_Clean is NaN"

wrapped = df['Title_Clean'].str.startswith('"') & df['Title_Clean'].str.endswith('"')
assert wrapped.sum() == 0, f"{wrapped.sum()} cleaned titles still wrapped in quotes"

starts_triple = df['Description_Clean'].fillna('').str.startswith('"""')
assert starts_triple.sum() == 0, f"{starts_triple.sum()} cleaned descriptions still start with triple quotes"

jira_left = df['Description_Clean'].fillna('').str.contains(r'\{code', regex=True)
assert jira_left.sum() == 0, f"{jira_left.sum()} cleaned descriptions still contain Jira code markup"

html_left = df['Description_Clean'].fillna('').str.contains(r'<[a-zA-Z/][^>]*>', regex=True)
assert html_left.sum() == 0, f"{html_left.sum()} cleaned descriptions still contain HTML tags"

assert df['Story_Point'].dtype.kind in 'fi', f"Story_Point dtype is {df['Story_Point'].dtype}"

flag_cols = [c for c in df.columns if c.startswith('flag_')]
for c in flag_cols:
    assert df[c].dtype == bool, f"{c} is not boolean, it is {df[c].dtype}"
print("All sanity checks passed.")
print(f"Final row count:    {len(df):,}")
print(f"Final column count: {len(df.columns)}")
print(f"Flag columns:       {len(flag_cols)}")

All sanity checks passed.
Final row count:    31,394
Final column count: 31
Flag columns:       10


In [10]:
# for the find the row that still has wrapping quotes after cleaning

still_wrapped_mask = (
    df['Title_Clean'].str.startswith('"')
    & df['Title_Clean'].str.endswith('"')
)

problem_rows = df[still_wrapped_mask][['Issue_Key', 'Project_Name', 'Title', 'Title_Clean']]

print(f"Rows still wrapped after cleaning: {len(problem_rows)}\n")
print("Raw Title vs Title_Clean:")
print("-" * 80)
for idx, row in problem_rows.iterrows():
    print(f"Row index: {idx}")
    print(f"  Issue_Key:   {row['Issue_Key']}")
    print(f"  Project:     {row['Project_Name']}")
    print(f"  Title raw:   {row['Title']!r}")
    print(f"  Title_Clean: {row['Title_Clean']!r}")
    print()

Rows still wrapped after cleaning: 1

Raw Title vs Title_Clean:
--------------------------------------------------------------------------------
Row index: 15087
  Issue_Key:   DM-18350
  Project:     Lsstcorp Data management
  Title raw:   '" ""Examine AuxTel lab data"""'
  Title_Clean: '""Examine AuxTel lab data""'



In [11]:
def strip_outer_quotes(s, max_iters=5):
    """Strip pairs of outer double quotes repeatedly."""
    for _ in range(max_iters):
        s = s.strip()
        if len(s) >= 2 and s.startswith('"') and s.endswith('"'):
            s = s[1:-1]
        else:
            break
    return s.strip()


def clean_title(text):
    if pd.isna(text):
        return text
    s = str(text).strip()
    s = strip_outer_quotes(s)
    s = HTML_TAG_RE.sub(' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def clean_description(text):
    if pd.isna(text):
        return text
    s = str(text).strip()

    m = TRIPLE_QUOTE_OUTER.match(s)
    if m is not None:
        s = m.group(1).strip()

    s = strip_outer_quotes(s)
    s = JIRA_MARKUP_RE.sub(' ', s)
    s = HTML_TAG_RE.sub(' ', s)
    s = CONTROL_CHARS_RE.sub('', s)
    s = WHITESPACE_RUN.sub(' ', s).strip()
    return s

df['Title_Clean']       = df['Title'].apply(clean_title)
df['Description_Clean'] = df['Description_Text'].apply(clean_description)
print("DM-18350 after fix:")
print(df.loc[15087, ['Title', 'Title_Clean']].to_dict())

still = (
    df['Title_Clean'].str.startswith('"')
    & df['Title_Clean'].str.endswith('"')
).sum()
print(f"\nTitles still wrapped: {still}")

DM-18350 after fix:
{'Title': '" ""Examine AuxTel lab data"""', 'Title_Clean': 'Examine AuxTel lab data'}

Titles still wrapped: 0


In [13]:
# Recompute text length columns and all flags using the freshly cleaned text.
df['title_char_count']       = df['Title_Clean'].fillna('').str.len()
df['title_word_count']       = df['Title_Clean'].fillna('').str.split().str.len()
df['description_char_count'] = df['Description_Clean'].fillna('').str.len()
df['description_word_count'] = df['Description_Clean'].fillna('').str.split().str.len()
df['flag_title_too_short']         = df['title_word_count'] < 3
df['flag_title_too_long']          = df['title_word_count'] > 20
df['flag_description_missing']     = df['description_char_count'] == 0
df['flag_description_too_short']   = (
    (df['description_word_count'] > 0) & (df['description_word_count'] < 10)
)
df['flag_description_markup_only'] = (
    (df['Description_Text'].fillna('').str.strip() != '')
    & (df['description_char_count'] == 0)
)

dup_key = (
    df['Project_Name'].fillna('')
    + '||'
    + df['Title_Clean'].fillna('')
    + '||'
    + df['Description_Clean'].fillna('')
)
df['is_duplicate_in_project']   = dup_key.duplicated(keep=False)
df['flag_duplicate_in_project'] = df['is_duplicate_in_project']

fibonacci = {0, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144}
df['flag_sp_non_fibonacci']      = df['Story_Point'].notna() & ~df['Story_Point'].isin(fibonacci)
df['flag_sp_high_scope_risk']    = df['Story_Point'] >= 13
df['flag_sp_extreme_scope_risk'] = df['Story_Point'] >= 40
df['flag_sp_missing']            = df['Story_Point'].isna()
print("Flag table after recompute:\n")
flag_cols = [c for c in df.columns if c.startswith('flag_')]
print(f"{'Flag':<38} {'True count':>12} {'Share':>10}")
print("-" * 65)
for col in flag_cols:
    n = int(df[col].sum())
    pct = n / len(df) * 100
    print(f"{col:<38} {n:>12,} {pct:>9.1f}%")

Flag table after recompute:

Flag                                     True count      Share
-----------------------------------------------------------------
flag_title_too_short                            845       2.7%
flag_title_too_long                             207       0.7%
flag_description_missing                      4,102      13.1%
flag_description_too_short                    2,282       7.3%
flag_description_markup_only                     35       0.1%
flag_sp_non_fibonacci                         7,500      23.9%
flag_sp_high_scope_risk                       1,094       3.5%
flag_sp_extreme_scope_risk                       75       0.2%
flag_sp_missing                               9,661      30.8%
flag_duplicate_in_project                       540       1.7%


In [15]:
mask = df['Description_Clean'].fillna('').str.startswith('"""')
problem_rows = df[mask][['Issue_Key', 'Project_Name', 'Description_Text', 'Description_Clean']]

print(f"Problem rows: {len(problem_rows)}\n")
for idx, row in problem_rows.head(5).iterrows():
    print(f"Row index: {idx}")
    print(f"  Issue_Key: {row['Issue_Key']}")
    print(f"  Raw       (first 200 chars): {str(row['Description_Text'])[:200]}")
    print(f"  Cleaned   (first 200 chars): {str(row['Description_Clean'])[:200]}")
    print(f"  Raw last 50 chars:           {str(row['Description_Text'])[-50:]}")
    print(f"  Cleaned last 50 chars:       {str(row['Description_Clean'])[-50:]}")
    print()

Problem rows: 15

Row index: 899
  Issue_Key: XD-2369
  Raw       (first 200 chars): """""""the meaning of the backtick has changed. The backtick now only does monospaced formatting, it does not escape the content. The migration guide walks you through the options:   http://asciidocto
  Cleaned   (first 200 chars): """"the meaning of the backtick has changed. The backtick now only does monospaced formatting, it does not escape the content. The migration guide walks you through the options: http://asciidoctor.org
  Raw last 50 chars:           sciidoctor/asciidoctor-gradle-plugin/issues/134"""
  Cleaned last 50 chars:       m/asciidoctor/asciidoctor-gradle-plugin/issues/134

Row index: 2244
  Issue_Key: XD-426
  Raw       (first 200 chars): """""""test the deployed module"""" sub-section uses curl."""
  Cleaned   (first 200 chars): """"test the deployed module"""" sub-section uses curl.
  Raw last 50 chars:            the deployed module"""" sub-section uses curl."""
  Cleaned last 50 c

In [16]:
# Final cleaning patch
LEADING_QUOTES_RE  = re.compile(r'^"+')
TRAILING_QUOTES_RE = re.compile(r'"+$')
def clean_description(text):
    if pd.isna(text):
        return text
    s = str(text).strip()
    m = TRIPLE_QUOTE_OUTER.match(s)
    if m is not None:
        s = m.group(1).strip()
    s = strip_outer_quotes(s)
    s = LEADING_QUOTES_RE.sub('', s)
    s = TRAILING_QUOTES_RE.sub('', s)
    s = s.strip()
    s = JIRA_MARKUP_RE.sub(' ', s)
    s = HTML_TAG_RE.sub(' ', s)
    s = CONTROL_CHARS_RE.sub('', s)
    s = WHITESPACE_RUN.sub(' ', s).strip()
    return s

df['Description_Clean'] = df['Description_Text'].apply(clean_description)
df['description_char_count'] = df['Description_Clean'].fillna('').str.len()
df['description_word_count'] = df['Description_Clean'].fillna('').str.split().str.len()

still_triple = df['Description_Clean'].fillna('').str.startswith('"""').sum()
still_lead   = df['Description_Clean'].fillna('').str.startswith('"').sum()
still_trail  = df['Description_Clean'].fillna('').str.endswith('"').sum()

print(f"Descriptions still starting with triple quotes: {still_triple}")
print(f"Descriptions still starting with any quote:     {still_lead}")
print(f"Descriptions still ending with any quote:       {still_trail}")
print("\nXD-426 before vs after:")
sample = df[df['Issue_Key'] == 'XD-426'].iloc[0]
print(f"  Raw     (first 100): {str(sample['Description_Text'])[:100]}")
print(f"  Cleaned (first 100): {str(sample['Description_Clean'])[:100]}")

Descriptions still starting with triple quotes: 0
Descriptions still starting with any quote:     0
Descriptions still ending with any quote:       0

XD-426 before vs after:
  Raw     (first 100): """""""test the deployed module"""" sub-section uses curl."""
  Cleaned (first 100): test the deployed module"""" sub-section uses curl.


In [18]:
# Problemmm need to inspect descriptions that still contain

mask = df['Description_Clean'].fillna('').str.contains(r'\{code', regex=True)
problem_rows = df[mask][['Issue_Key', 'Project_Name', 'Description_Clean']]

print(f"Problem rows: {len(problem_rows)}\n")
for idx, row in problem_rows.head(5).iterrows():
    print(f"Row index: {idx}, Issue_Key: {row['Issue_Key']}")
    text = str(row['Description_Clean'])
    pos = text.find('{code')
    start = max(0, pos - 40)
    end   = min(len(text), pos + 200)
    print(f"  Context: ...{text[start:end]}...")
    print()

Problem rows: 41

Row index: 5744, Issue_Key: FAB-16830
  Context: ... specify a value. For example, decoding {code:yaml} --- stringOne: my special string where {{Numbers}} is defined as: {code:go} type Numbers struct { NumberOne int `yaml:""""numberOne"""" default:""""1""""` NumberTwo int `yaml:""""numberTwo...

Row index: 6902, Issue_Key: FAB-8496
  Context: ...ect. If I modify the {{env.sh}} script: {code:none} # Names of the orderer organizations ORDERER_ORGS=""""eu"""" # Names of the peer organizations PEER_ORGS=""""fr ge it sp"""" and {code:none} writeIntermediateCA() ... environment: - FABRIC...

Row index: 8725, Issue_Key: DM-26713
  Context: ...is slower to create its representation: {code:python} from lsst.pipe.tasks.calibrate import CalibrateConfig from io import StringIO import yaml config = CalibrateConfig() # write only %timeit u = yaml.dump(config) %timeit strio = StringIO()...

Row index: 8955, Issue_Key: DM-26439
  Context: ...Ref. For example, running this command: {c

In [19]:
# Try to stronger Jira markup handling
JIRA_CODE_PAIR_RE     = re.compile(r'\{code[^\}]*\}.*?\{code\}',     re.DOTALL)
JIRA_NOFORMAT_PAIR_RE = re.compile(r'\{noformat\}.*?\{noformat\}',   re.DOTALL)
JIRA_QUOTE_PAIR_RE    = re.compile(r'\{quote\}.*?\{quote\}',         re.DOTALL)
JIRA_PANEL_PAIR_RE    = re.compile(r'\{panel[^\}]*\}.*?\{panel\}',   re.DOTALL)
JIRA_STRAY_TAG_RE = re.compile(r'\{(?:code|noformat|quote|panel)[^\}]*\}')
JIRA_INLINE_BRACE_RE = re.compile(r'\{\{([^\}]+)\}\}')

def clean_description(text):
    if pd.isna(text):
        return text
    s = str(text).strip()
    m = TRIPLE_QUOTE_OUTER.match(s)
    if m is not None:
        s = m.group(1).strip()
    s = strip_outer_quotes(s)
    s = LEADING_QUOTES_RE.sub('', s)
    s = TRAILING_QUOTES_RE.sub('', s)
    s = s.strip()
    s = JIRA_CODE_PAIR_RE.sub(' ',     s)
    s = JIRA_NOFORMAT_PAIR_RE.sub(' ', s)
    s = JIRA_QUOTE_PAIR_RE.sub(' ',    s)
    s = JIRA_PANEL_PAIR_RE.sub(' ',    s)
    s = JIRA_STRAY_TAG_RE.sub(' ', s)
    s = JIRA_INLINE_BRACE_RE.sub(r'\1', s)
    s = HTML_TAG_RE.sub(' ', s)
    s = CONTROL_CHARS_RE.sub('', s)
    s = WHITESPACE_RUN.sub(' ', s).strip()
    return s

df['Description_Clean'] = df['Description_Text'].apply(clean_description)
df['description_char_count'] = df['Description_Clean'].fillna('').str.len()
df['description_word_count'] = df['Description_Clean'].fillna('').str.split().str.len()
still_code   = df['Description_Clean'].fillna('').str.contains(r'\{code',     regex=True).sum()
still_brace  = df['Description_Clean'].fillna('').str.contains(r'\{\{',       regex=True).sum()
print(f"Descriptions still containing '{{code': {still_code}")
print(f"Descriptions still containing '{{{{':   {still_brace}")

Descriptions still containing '{code': 0
Descriptions still containing '{{':   29


In [20]:
mask = df['Description_Clean'].fillna('').str.contains(r'\{\{', regex=True)
problem_rows = df[mask][['Issue_Key', 'Project_Name', 'Description_Clean']]
print(f"Problem rows: {len(problem_rows)}\n")
for idx, row in problem_rows.head(5).iterrows():
    print(f"Row index: {idx}, Issue_Key: {row['Issue_Key']}")
    text = str(row['Description_Clean'])
    pos = text.find('{{')
    start = max(0, pos - 50)
    end   = min(len(text), pos + 200)
    print(f"  Context: ...{text[start:end]}...")
    print()

Problem rows: 29

Row index: 873, Issue_Key: XD-2418
  Context: ...nd add the corresponding attribute to the element {{async=""""$\{async\}""""}}...

Row index: 3795, Issue_Key: MXNET-1429
  Context: ...Current constructor where dev_type is a {{mx.CONTEXT_TYPE} enum. I want to add a constructor where `dev_type` is just a string const: `:cpu`, `:gpu` or `:cpu_pinned`....

Row index: 3889, Issue_Key: MXNET-569
  Context: ...as', shape=(num_filter,), init=mx.init.Zero()) or {{{ """"op"""": """"null"""", """"name"""": """"conv_bias"""", """"attrs"""": \{ """"__init__"""": """"[\""""zero\"""", {}]"""", """"__shape__"""": """"(64,)"""" }, """"inputs"""": [] }}}...

Row index: 4045, Issue_Key: ALOY-221
  Context: ... case the developer does not have 2.7+ installed, {{}}. This method, unfortunately, does not give the full error log subprocess.check_output does. We need to implement a method in the ti.alloy compiler plugin that allows for solid error output on pyt...

Row index: 5629, Issue_Key

In [21]:
JIRA_BRACE_RUN_OPEN  = re.compile(r'\{\{+')
JIRA_BRACE_RUN_CLOSE = re.compile(r'\}\}+')

def clean_description(text):
    if pd.isna(text):
        return text
    s = str(text).strip()
    m = TRIPLE_QUOTE_OUTER.match(s)
    if m is not None:
        s = m.group(1).strip()

    s = strip_outer_quotes(s)
    s = LEADING_QUOTES_RE.sub('', s)
    s = TRAILING_QUOTES_RE.sub('', s)
    s = s.strip()
    s = JIRA_CODE_PAIR_RE.sub(' ',     s)
    s = JIRA_NOFORMAT_PAIR_RE.sub(' ', s)
    s = JIRA_QUOTE_PAIR_RE.sub(' ',    s)
    s = JIRA_PANEL_PAIR_RE.sub(' ',    s)
    s = JIRA_STRAY_TAG_RE.sub(' ', s)
    s = JIRA_INLINE_BRACE_RE.sub(r'\1', s)
    s = JIRA_BRACE_RUN_OPEN.sub(' ',  s)
    s = JIRA_BRACE_RUN_CLOSE.sub(' ', s)
    s = HTML_TAG_RE.sub(' ', s)
    s = CONTROL_CHARS_RE.sub('', s)
    s = WHITESPACE_RUN.sub(' ', s).strip()
    return s

df['Description_Clean'] = df['Description_Text'].apply(clean_description)
df['description_char_count'] = df['Description_Clean'].fillna('').str.len()
df['description_word_count'] = df['Description_Clean'].fillna('').str.split().str.len()
still_code  = df['Description_Clean'].fillna('').str.contains(r'\{code', regex=True).sum()
still_brace = df['Description_Clean'].fillna('').str.contains(r'\{\{',   regex=True).sum()
still_close = df['Description_Clean'].fillna('').str.contains(r'\}\}',   regex=True).sum()
print(f"Still containing '{{code': {still_code}")
print(f"Still containing '{{{{':   {still_brace}")
print(f"Still containing '}}}}':   {still_close}")

Still containing '{code': 0
Still containing '{{':   0
Still containing '}}':   0


In [23]:
# Save the cleaned data to data/processed/.
OUT_PATH = Path('../data/processed/02_user_stories_clean.csv')
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False, encoding='utf-8')
import os
size_mb = os.path.getsize(OUT_PATH) / 1024**2

print(f"Saved: {OUT_PATH}")
print(f"Rows:  {len(df):,}")
print(f"Cols:  {len(df.columns)}")
print(f"Size:  {size_mb:.1f} MB on disk")

Saved: ..\data\processed\02_user_stories_clean.csv
Rows:  31,394
Cols:  31
Size:  40.2 MB on disk


## Summary

**What this notebook did:**
- Loaded 31,394 raw user stories from `data/interim/01_user_stories_raw.csv`.
- Counted text artifacts in the raw data before touching anything.
- Wrote two cleaning functions (`clean_title` and `clean_description`) and applied them.
- Created `Title_Clean` and `Description_Clean` columns. Original columns are kept untouched for audit.
- Detected exact duplicates within each project (same Title plus Description plus Project).
- Added 10 boolean quality flags. Stories with issues are flagged, not dropped.
- Ran 8 sanity assertions to catch silent cleaning failures.
- Saved the canonical clean file to `data/processed/02_user_stories_clean.csv`.

**Cleaning rules applied:**

| Problem | Rows affected | Action |
|---|---:|---|
| Title wrapped in double quotes | 31,394 (100%) | Stripped outer quote pairs, including nested ones. |
| Description starts with triple quotes | 27,327 (87%) | Stripped triple quote wrap and leftover edge quotes. |
| Description contains HTML tags | 2,389 (7.6%) | Tags removed, surrounding text kept. |
| Description contains Jira markup | 415 (1.3%) plus stray tags | Paired blocks removed with their content. Stray tags removed. Inline `{{ref}}` unwrapped. |
| Description contains control chars | 5 | Removed silently. |

**Quality flags produced:**

| Flag | True count | Share |
|---|---:|---:|
| flag_title_too_short | 845 | 2.7% |
| flag_title_too_long | 207 | 0.7% |
| flag_description_missing | 4,102 | 13.1% |
| flag_description_too_short | 2,282 | 7.3% |
| flag_description_markup_only | 35 | 0.1% |
| flag_sp_non_fibonacci | 7,500 | 23.9% |
| flag_sp_high_scope_risk | 1,094 | 3.5% |
| flag_sp_extreme_scope_risk | 75 | 0.2% |
| flag_sp_missing | 9,661 | 30.8% |
| flag_duplicate_in_project | 540 | 1.7% |

**New findings from this notebook (added to docs/findings.md):**
- 35 user stories contained only markup, no natural language. Cleaning revealed this.
- 540 stories are exact duplicates inside the same project. One example: four identical Spring XD stories created within three minutes of each other.
- 23.9% of all stories carry a non Fibonacci Story Point estimate. This is about a third of the stories that have any estimate at all.

**What is next:** notebook 03 reads `02_user_stories_clean.csv`, builds quality features for scoring (clarity, completeness, testability signals) and aligns them to the QUS framework.